**Please set up your credentials JSON as GCP_CREDENTIALS secrets**

In [1]:
import json
import os
from pathlib import Path

credentials_path = Path("gcp-key.json")
with credentials_path.open(encoding="utf-8") as credentials_file:
    credentials = json.load(credentials_file)

# Do not expose GCP JSON credentials as global dlt credentials.
os.environ.pop("DESTINATION__CREDENTIALS", None)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credentials_path.resolve())
print(f"Loaded GCP credentials from {credentials_path.resolve()}")

Loaded GCP credentials from C:\Users\letie\Documents\GitHub\docker-workshop\03-data-warehouse\gcp-key.json


In [2]:
# Install for production in the active notebook kernel
%pip install "dlt[bigquery,gs]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Install for local testing in the active notebook kernel
%pip install "dlt[duckdb]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import dlt
import requests
import pandas as pd
from dlt.destinations import filesystem
from io import BytesIO

In [ ]:
# Define a dlt source to download and process CSV files as resources
@dlt.source(name="rides")
def download_parquet():
    prefix = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download"
    for taxi in ["yellow", "green"]:
        for year in [2019, 2020]:
            for month in range(1, 13):
                month_code = f"{month:02d}"
                file_name = f"{taxi}_tripdata_{year}-{month_code}"
                url = f"{prefix}/{taxi}/{taxi}_tripdata_{year}-{month_code}.csv.gz"
                response = requests.get(url, timeout=60)
                response.raise_for_status()
                df = pd.read_csv(BytesIO(response.content), compression="gzip", dtype={"store_and_fwd_flag": "string"})
                yield dlt.resource(df, name=file_name)


BUCKET_URL = os.environ.get("BUCKET_URL", "gs://dezoomcamp_hw3_vinh")

pipeline = dlt.pipeline(
    pipeline_name="rides_gcs",
    destination=filesystem(
        bucket_url=BUCKET_URL,
        credentials=credentials,
        layout="{schema_name}/{table_name}.{ext}",
    ),
    dataset_name="rides_20192020",
)

load_info = pipeline.run(download_parquet(), loader_file_format="csv")
print(load_info)

C:\Users\letie\AppData\Local\Temp\ipykernel_20688\4252939558.py:13: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(response.content), compression="gzip")
C:\Users\letie\AppData\Local\Temp\ipykernel_20688\4252939558.py:13: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(response.content), compression="gzip")
C:\Users\letie\AppData\Local\Temp\ipykernel_20688\4252939558.py:13: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(response.content), compression="gzip")
C:\Users\letie\AppData\Local\Temp\ipykernel_20688\4252939558.py:13: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BytesIO(response.content), comp

KeyboardInterrupt: 

Ingesting data to Database

In [6]:
# Define a dlt resource to download and process Parquet files as a single table
@dlt.resource(name="rides", write_disposition="replace")
def download_parquet_duckdb():
    prefix = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata"

    for month in range(1, 7):
        url = f"{prefix}_2024-0{month}.parquet"
        response = requests.get(url)
        response.raise_for_status()

        df = pd.read_parquet(BytesIO(response.content))
        yield df


pipeline = dlt.pipeline(
    pipeline_name="rides_duckdb",
    destination="duckdb",
    dataset_name="rides_dataset",
)

info = pipeline.run(download_parquet_duckdb)
print(info)

Pipeline rides_duckdb load step finished in 2.75 seconds
1 load package(s) were loaded to destination duckdb and into dataset rides_dataset
The duckdb destination used duckdb:///c:\Users\letie\Documents\GitHub\docker-workshop\03-data-warehouse\rides_duckdb.duckdb location to store data
Load package 1787141540.3656037 is LOADED and contains no failed jobs


In [7]:
import duckdb

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")

# Set search path to the dataset
conn.sql(f"SET search_path = '{pipeline.dataset_name}'")

# Describe the dataset to see loaded tables
res = conn.sql("DESCRIBE").df()
print(res)

       database         schema                 name  \
0  rides_duckdb  rides_dataset           _dlt_loads   
1  rides_duckdb  rides_dataset  _dlt_pipeline_state   
2  rides_duckdb  rides_dataset         _dlt_version   
3  rides_duckdb  rides_dataset                rides   

                                        column_names  \
0  [load_id, schema_name, status, inserted_at, sc...   
1  [version, engine_version, pipeline_name, state...   
2  [version, engine_version, inserted_at, schema_...   
3  [vendor_id, tpep_pickup_datetime, tpep_dropoff...   

                                        column_types  temporary  
0  [VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...      False  
1  [BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...      False  
2  [BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...      False  
3  [INTEGER, TIMESTAMP WITH TIME ZONE, TIMESTAMP ...      False  


In [8]:
# provide a resource name to query a table of that name
with pipeline.sql_client() as client:
    with client.execute_query(f"SELECT count(1) FROM rides") as cursor:
        data = cursor.df()
print(data)

   count(1)
0  20332093
